# SPC 與 ML 整合：預測結果轉資料庫警報狀態設計

- 目標：精通機器學習預測（ML Prediction）與統計製程管制（SPC）規則的動態整合。學會如何利用歷史 SPC 管制界限（UCL/LCL）對 ML 的即時預測結果進行「警報狀態等級分類」，並串接 SQLAlchemy 資料庫，更新 alert_status 欄位，完成閉環式自動化警報控制系統。


## 1. ML 與 SPC 整合的商業與物理邏輯

- 在半導體自動化產線中，僅僅獲得機器學習模型的預測良率（例如：模型預測這片晶圓良率為 93.5%）是不夠的。我們必須將預測值與產線歷史的 SPC 統計管制界限 進行交叉比對：
    - 預測值低於 LCL（下管制界限）：代表模型預判製程即將失控，必須立刻觸發 CRITICAL（嚴重警報） 並攔截產品。
    - 預測值在 LCL 至 1-Sigma 之間：代表有下滑趨勢，觸發 WARNING（警告）。
    - 預測值在管制界限內：維持 NORMAL（正常）。
- 這項設計對應了資料庫 YieldPredictionRecord 中 alert_status 欄位的動態邏輯設計。


## 2. SPC 管制規則核心邏輯與 Alert 轉換實作

- 我們將編寫一個 SPCAlertEngine 類別，它能依據歷史基準值計算標準差，並將 ML 預測值對應到相應的警報層級。


In [ ]:
import numpy as np
import pandas as pd


class SPCAlertEngine:
    """SPC 與 機器學習預測結果整合警報引擎"""

    def __init__(self, historical_yields: np.ndarray):
        # 1. 依據歷史良率大數據，計算基準中心線與標準差
        self.cl = np.mean(historical_yields)
        self.sigma = np.std(historical_yields, ddof=1)

        # 2. 定義 3-Sigma 傳統 SPC 管制界限 (Western Electric 規則基礎)
        self.ucl = self.cl + 3 * self.sigma
        self.lcl = self.cl - 3 * self.sigma

        # 3. 定義 1-Sigma 警告界限
        self.warning_lcl = self.cl - 1.5 * self.sigma

    def evaluate_prediction(self, predicted_yield: float) -> str:
        """將 ML 預測良率轉換為 SPC 警報狀態字串"""
        if predicted_yield < self.lcl:
            return "CRITICAL"  # 超出 3-Sigma 下限，代表製程即將嚴重失控
        elif predicted_yield < self.warning_lcl:
            return "WARNING"  # 超出 1.5-Sigma，代表有異常趨勢，需預警
        else:
            return "NORMAL"  # 安全範圍內


# ---- 實作測試警報引擎 ----
# 模擬 50 片歷史優良晶圓的真實良率數據（基準點）
np.random.seed(100)
history_data = np.random.normal(loc=98.2, scale=0.4, size=50)

alert_engine = SPCAlertEngine(history_data)
print("=" * 50)
print(f"📊 SPC 基準建立：中心線(CL) = {alert_engine.cl:.2f}%")
print(f"⚠️ SPC 警告界限 (1.5-Sigma LCL) = {alert_engine.warning_lcl:.2f}%")
print(f"🚨 SPC 管制下限 (3-Sigma LCL)  = {alert_engine.lcl:.2f}%")
print("=" * 50)

# 模擬機器學習模型對新批次 3 片晶圓的預測結果
mock_predictions = [98.0, 97.1, 95.8]

for i, p_yield in enumerate(mock_predictions):
    status = alert_engine.evaluate_prediction(p_yield)
    print(f"🔮 Wafer #{i + 1} -> ML 預測良率: {p_yield}% | 轉化 SPC 狀態: 【{status}】")


## 3. 串接 SQLAlchemy 資料庫動態寫入與更新 alert_status

- 實戰場景：當自動化 Pipeline 執行時，主程式呼叫 ML 模型進行推理，獲得預測良率後，利用上述引擎判定狀態，最後使用 ORM 安全地將狀態同步寫入資料庫的 alert_status 欄位中。


In [ ]:
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker

# 1. 建立記憶體 SQLite 資料庫供集成測試
db_engine = create_engine("sqlite:///:memory:", echo=False)
Base = declarative_base()


# 2. 定義良率預測紀錄表模型 (呼應 3.1 設計)
class YieldPredictionRecord(Base):
    __tablename__ = "yield_prediction_records"

    id = Column(Integer, primary_key=True, autoincrement=True)
    wafer_id_str = Column(String(50), nullable=False)
    predicted_yield = Column(Float, nullable=False)
    alert_status = Column(
        String(20), default="NORMAL"
    )  # 核心欄位：NORMAL / WARNING / CRITICAL
    updated_at = Column(DateTime, default=datetime.utcnow)


Base.metadata.create_all(db_engine)
Session = sessionmaker(bind=db_engine)
session = Session()

# ===================================================
# 🔄 整合自動化 Pipeline 模擬
# ===================================================
print("\n>>> 啟動 ML × SPC 資料庫同步 Pipeline...")

# 模擬新晶圓上線測試
new_wafers_data = {
    "WAFER_LOT99_01": 98.3,
    "WAFER_LOT99_02": 97.2,  # 觸發警告
    "WAFER_LOT99_03": 94.5,  # 觸發嚴重失控
}

for w_id, ml_pred in new_wafers_data.items():
    # A. 透過 SPC 引擎判定狀態
    calculated_status = alert_engine.evaluate_prediction(ml_pred)

    # B. 建立資料庫 ORM 物件
    record = YieldPredictionRecord(
        wafer_id_str=w_id, predicted_yield=ml_pred, alert_status=calculated_status
    )
    session.add(record)

session.commit()
print("✅ 所有 ML 預測與 SPC 警報狀態已成功持久化至資料庫！")

# 3. 驗證資料庫內容
print("\n🔍 查詢資料庫中需要【立刻攔截或派工檢查】的晶圓紀錄：")
flagged_records = (
    session.query(YieldPredictionRecord)
    .filter(YieldPredictionRecord.alert_status.in_(["WARNING", "CRITICAL"]))
    .all()
)

for r in flagged_records:
    print(
        f"🚨 [Alert] 晶圓: {r.wafer_id_str} | 預測良率: {r.predicted_yield}% | 狀態: {r.alert_status} | 時間: {r.updated_at}"
    )

session.close()


- 總結：在我們團隊的架構設計中，機器學習模型不應該是獨立的孤島。我實作了 SPC 與 ML 整合的自動化警報機制。當資料管道清洗完數據後，ML 模型（如 XGBoost）會即時預測該片晶圓的預期良率。此時，我的 SPCAlertEngine 模組會動態載入該產品線的歷史良率分佈基準線，並依據統計學標準差將預測結果歸類。如果預測值低於 3-Sigma 管制界限（LCL），程式會自動將資料庫的 alert_status 欄位標記為 CRITICAL。這套架構的好處在於，後續的自動化派工系統或看板只要訂閱資料庫的這個欄位，就能在產品還沒真正出廠前，就先預警攔截潛在的不良品，完美體現了資料科學在半導體自動化控制上的軟實力。
